In [1]:
import cv2
from ultralytics import YOLO

import pytesseract

import time


In [2]:
# classe_choisie = "fighter_white"

In [3]:
model_name = "y12x_320px_15patience" 
score_board = "_score_board"

model_path = f"../../models/fine_tuned_models{score_board}/{model_name}/weights/best.pt"

# model_path = f"../../models/pretrainned_models/yolo12x.pt"

In [4]:
model = YOLO(model_path, task="detect")

In [5]:
source = '../../outputs/video_extract_img/video_test_cut.mp4'

path_video_output = '../../outputs/video_annotees'

In [6]:
WINDOW_NAME = 'figther_tracking'
WINDOW_WIDTH = 1000   # Nouvelle largeur souhaitée
WINDOW_HEIGHT = 800  # Nouvelle hauteur souhaitée
WINDOW_POS_X = 1900   # Position horizontale sur l'écran
WINDOW_POS_Y = 100   # Position verticale sur l'écran

In [7]:
# Open the video file
video_path = source
cap = cv2.VideoCapture(video_path)

# Variable pour contrôler la fréquence de détection
last_detection_time = 0.0  # on stocke le temps de la dernière détection
DETECTION_INTERVAL = 0.2   # en secondes, ici 1 s

# Pour stocker l'image annotée la plus récente
annotated_frame = None
text ="_"

# Loop through the video frames
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()
    
    if not success:
        break

        # Run  tracking on the frame, persisting tracks between frames
    results = model(
                                source=frame,
                                conf=0.75, 
                                iou=0.05, 
                                # classes=1, 
                                max_det=1
                            )
    
    # Récupère le temps actuel
    current_time = time.time()

    if current_time - last_detection_time >= DETECTION_INTERVAL:
        # Met à jour le temps de la dernière détection
        last_detection_time = current_time    


        # Récupère les infos de détection
        boxes = results[0].boxes
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            # marge autour de la boîte englobante
            height, width, _ = frame.shape

            margin = 10
            x1 = max(0, x1 - margin)
            y1 = max(0, y1 - margin)
            x2 = min(width,  x2 + margin)
            y2 = min(height, y2 + margin)

            cropped = frame[y1:y2, x1:x2]


            # 3) Prétraitement de l’image pour l’OCR
            gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)

            # Binarisation
            _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)
            # (Optionnel) Fermeture morphologique
            kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1,1))
            thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

            # Option : redimensionner si c'est petit
            h_cropped, w_cropped = thresh.shape
            if w_cropped < 100 or h_cropped < 40:
                scale = 2
                thresh = cv2.resize(thresh, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)


            # 4) OCR avec Tesseract
            config = (
                r'--oem 3 '                 # moteur LSTM
                r'--psm 11 '                 # suppose une ligne de texte
                r'tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789:'
            )
            text = pytesseract.image_to_string(thresh, config=config).strip()

            print("Texte détecté :", text, "x1:", x1, "y1:", y1, "x2:", x2, "y2:", y2)



        # ----- Fin du bloc de détection -----

    else:
        # Si on n'effectue pas de détection sur cette frame,
        # on peut simplement copier la frame « brute » pour l'affichage
        # ou réutiliser la dernière image annotée si vous préférez.
        if annotated_frame is None:
            annotated_frame = frame.copy()
        else:
            # Option 1 : réutiliser la dernière image annotée
            # annotated_frame = annotated_frame

            # Option 2 : ou bien afficher la frame en direct (non annotée)
            annotated_frame = frame.copy()


                # Visualize the results on the frame
    annotated_frame = results[0].plot()

    # (Optionnel) Afficher le texte sur le frame
    # cv2.putText(annotated_frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Affichage du texte sur l'image
    cv2.putText(
        annotated_frame,
        text,         # Le texte à afficher
        (150, 300),                   # Coordonnées (x, y) dans l'image
        cv2.FONT_HERSHEY_SIMPLEX, # Police (exemple : FONT_HERSHEY_SIMPLEX)
        1,                        # Taille du texte (scale)
        (0, 255, 0),              # Couleur (B, G, R) ici vert
        2                         # Épaisseur du trait
    )


    # Affichage du texte sur l'image
    cv2.putText(
        annotated_frame,
        "x1 y1",         # Le texte à afficher
        (x1, y1),                   # Coordonnées (x, y) dans l'image
        cv2.FONT_HERSHEY_SIMPLEX, # Police (exemple : FONT_HERSHEY_SIMPLEX)
        1,                        # Taille du texte (scale)
        (0, 255, 0),              # Couleur (B, G, R) ici vert
        2                         # Épaisseur du trait
    )

        # Affichage du texte sur l'image
    cv2.putText(
        annotated_frame,
        "x2, y2",         # Le texte à afficher
        (x2, y2),                   # Coordonnées (x, y) dans l'image
        cv2.FONT_HERSHEY_SIMPLEX, # Police (exemple : FONT_HERSHEY_SIMPLEX)
        1,                        # Taille du texte (scale)
        (0, 255, 0),              # Couleur (B, G, R) ici vert
        2                         # Épaisseur du trait
    )

    # Crée et configure la fenêtre
    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(WINDOW_NAME, WINDOW_WIDTH, WINDOW_HEIGHT)
    cv2.moveWindow(WINDOW_NAME, WINDOW_POS_X, WINDOW_POS_Y)

    # Affiche le résultat (soit la frame annotée, soit la frame brute)
    cv2.imshow(WINDOW_NAME, annotated_frame)

    # Quitte si on appuie sur "q"
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# Release the video capture object and close the display window
cap.release()
cv2.destroyAllWindows()



0: 192x320 1 incrus_score_board, 93.3ms
Speed: 4.2ms preprocess, 93.3ms inference, 110.1ms postprocess per image at shape (1, 3, 192, 320)
Texte détecté :  x1: 135 y1: 26 x2: 390 y2: 226

0: 192x320 1 incrus_score_board, 28.5ms
Speed: 1.1ms preprocess, 28.5ms inference, 1.5ms postprocess per image at shape (1, 3, 192, 320)
Texte détecté :  x1: 135 y1: 26 x2: 390 y2: 226

0: 192x320 1 incrus_score_board, 28.2ms
Speed: 0.8ms preprocess, 28.2ms inference, 1.5ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 1 incrus_score_board, 31.5ms
Speed: 0.9ms preprocess, 31.5ms inference, 1.6ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 1 incrus_score_board, 28.0ms
Speed: 0.8ms preprocess, 28.0ms inference, 1.9ms postprocess per image at shape (1, 3, 192, 320)
Texte détecté :  x1: 134 y1: 25 x2: 390 y2: 227

0: 192x320 1 incrus_score_board, 27.6ms
Speed: 0.8ms preprocess, 27.6ms inference, 2.0ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 1 incrus_score